In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/herg_hackathon'

herg_sequence = "MPVRRGHVAPQNTFLDTIIRKFEGQSRKFIIANARVENCAVIYCNDGFCELCGYSRAEVMQRPCTCDFLHGPRTQRRAAAQIAQALLGAEERKVEIAFYRKDGSCFLCLVDVVPVKNEDGAVIMFILNFEVVMEKDMVGSPAHDTNHRGPPTSWLAPGRAKTFRLKLPALLALTARESSVRSGGAGGAGAPGAVVVDVDLTPAAPSSESLALDEVTAMDNHVAGLGPAEERRALVGPGSPPRSAPGQLPSPRAHSLNPDASGSSCSLARTRSRESCASVRRASSADDIEAMRAGVLPPPPRHASTGAMHPLRSGLLNSTSDSDLVRYRTISKIPQITLNFVDLKGDPFLASPTSDREIIAPKIKERTHNVTEKVTQVLSLGADVLPEYKLQAPRIHRWTILHYSPFKAVWDWLILLLVIYTAVFTPYSAAFLLKETEEGPPATECGYACQPLAVVDLIVDIMFIVDILINFRTTYVNANEEVVSHPGRIAVHYFKGWFLIDMVAAIPFDLLIFGSGSEELIGLLKTARLLRLVRVARKLDRYSEYGAAVLFLLMCTFALIAHWLACIWYAIGNMEQPHMDSRIGWLHNLGDQIGKPYNSSGLGGPSIKDKYVTALYFTFSSLTSVGFGNVSPNTNSEKIFSICVMLIGSLMYASIFGNVSAIIQRLYSGTARYHTQMLRVREFIRFHQIPNPLRQRLEEYFQHAWSYTNGIDMNAVLKGFPECLQADICLHLNRSLLQHCKPFRGATKGCLRALAMKFKTTHAPPGDTLVHAGDLLTALYFISRGSIEILRGDVVVAILGKNDIFGEPLNLYARPGKSNGDVRALTYCDLHKIHRDDLLEVLDMYPEFSDHFWSSLEITFNLRDTNMIPGSPGSTELEGGFSRQRKRKLSFRRRTDKDTEQPGEVSALGPGRAGAGPSSRGRPGGPWGESPSSGPSSPESSEDEGPGRSSSPLRLVPFSSPRPPGEPPGGEPLMEDCEKSSDTCNPLSGAFSGVSNIFSFWGDSRGRQYQELPRCPAPTPSLLNIPLSSPGRRPRGDVESRLDALQRQLNRLETRLSADMATVLQLLQRQMTLVPPAYSAVTTPGPGPTSTSPLLPVSPLPTLTLDSLSQVSQFMACEELPPGAPELPQEGPTRRLSLPGQLGALTSQPLHRHGSDPGS"

print(f"Sequence length check: {len(herg_sequence)}")

with open(f"{SAVE_DIR}/herg_protein_Q12809.fasta", "w") as f:
    f.write(f">sp|Q12809|KCNH2_HUMAN\n{herg_sequence}\n")
print("Saved herg_protein_Q12809.fasta")

Mounted at /content/drive
Sequence length check: 1159
Saved herg_protein_Q12809.fasta


In [ ]:
from collections import Counter
import numpy as np
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/herg_hackathon'

# Standard amino acid property scales
hydrophobicity = {  # Kyte-Doolittle scale
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}
charge = {  # net charge at physiological pH
    'D': -1, 'E': -1, 'K': 1, 'R': 1, 'H': 0.1
}

def get_protein_features(sequence):
    length = len(sequence)
    counts = Counter(sequence)

    # Amino acid composition (%)
    aa_composition = {f"AA_{aa}": counts.get(aa, 0) / length for aa in "ACDEFGHIKLMNPQRSTVWY"}

    # Aggregate physicochemical properties
    avg_hydrophobicity = np.mean([hydrophobicity.get(r, 0) for r in sequence])
    net_charge = sum([charge.get(r, 0) for r in sequence])
    basic_residue_frac = (counts.get('K', 0) + counts.get('R', 0)) / length
    acidic_residue_frac = (counts.get('D', 0) + counts.get('E', 0)) / length
    aromatic_frac = (counts.get('F', 0) + counts.get('W', 0) + counts.get('Y', 0)) / length

    features = {
        "protein_length": length,
        "protein_avg_hydrophobicity": avg_hydrophobicity,
        "protein_net_charge": net_charge,
        "protein_basic_frac": basic_residue_frac,
        "protein_acidic_frac": acidic_residue_frac,
        "protein_aromatic_frac": aromatic_frac,
        **aa_composition
    }
    return features

with open(f"{SAVE_DIR}/herg_protein_Q12809.fasta") as f:
    lines = f.read().strip().split("\n")
    herg_sequence = "".join(lines[1:])

protein_features = get_protein_features(herg_sequence)
print(f"Generated {len(protein_features)} protein-level features")
for k, v in list(protein_features.items())[:10]:
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Save as a single-row reference (since all compounds share this one target)
protein_df = pd.DataFrame([protein_features])
protein_df.to_csv(f"{SAVE_DIR}/herg_protein_features.csv", index=False)
print(f"\nSaved herg_protein_features.csv")

Generated 26 protein-level features
  protein_length: 1159
  protein_avg_hydrophobicity: -0.1728
  protein_net_charge: 7.8000
  protein_basic_frac: 0.1035
  protein_acidic_frac: 0.0992
  protein_aromatic_frac: 0.0742
  AA_A: 0.0802
  AA_C: 0.0207
  AA_D: 0.0475
  AA_E: 0.0518

Saved herg_protein_features.csv


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score)
import joblib

SAVE_DIR = '/content/drive/MyDrive/herg_hackathon'

# Load existing ligand features
fp_matrix = np.load(f"{SAVE_DIR}/fp_matrix.npy")
meta = pd.read_csv(f"{SAVE_DIR}/herg_featurized_meta.csv")
desc = pd.read_csv(f"{SAVE_DIR}/descriptors.csv")
protein_df = pd.read_csv(f"{SAVE_DIR}/herg_protein_features.csv")

X_ligand = np.hstack([fp_matrix, desc.values])
y_class = meta["label"].values

train_mask = (meta["split"] == "train").values
test_mask = (meta["split"] == "test").values

# Broadcast the single protein feature vector to every compound row
protein_vec = protein_df.values[0]  # shape (26,)
protein_matrix = np.tile(protein_vec, (X_ligand.shape[0], 1))  # shape (n_compounds, 26)

# Concatenate ligand + protein features
X_combined = np.hstack([X_ligand, protein_matrix])

print(f"Ligand-only feature shape: {X_ligand.shape}")
print(f"Combined (ligand+protein) feature shape: {X_combined.shape}")

X_train_combined = X_combined[train_mask]
X_test_combined = X_combined[test_mask]
y_train = y_class[train_mask]
y_test = y_class[test_mask]

print("\nTraining Random Forest with ligand+protein features...")
rf_combined = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
rf_combined.fit(X_train_combined, y_train)

probs = rf_combined.predict_proba(X_test_combined)[:, 1]
preds = rf_combined.predict(X_test_combined)

print("\nRandom Forest (Ligand + Protein Features)")
print(f"  Accuracy:  {accuracy_score(y_test, preds):.4f}")
print(f"  Precision: {precision_score(y_test, preds):.4f}")
print(f"  Recall:    {recall_score(y_test, preds):.4f}")
print(f"  F1:        {f1_score(y_test, preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, probs):.4f}")
print(f"  PR-AUC:    {average_precision_score(y_test, probs):.4f}")

joblib.dump(rf_combined, f"{SAVE_DIR}/model_rf_ligand_protein.pkl")
print("\nModel saved: model_rf_ligand_protein.pkl")

Ligand-only feature shape: (16159, 2056)
Combined (ligand+protein) feature shape: (16159, 2082)

Training Random Forest with ligand+protein features...

Random Forest (Ligand + Protein Features)
  Accuracy:  0.7762
  Precision: 0.7840
  Recall:    0.8387
  F1:        0.8104
  ROC-AUC:   0.8664
  PR-AUC:    0.8957

Model saved: model_rf_ligand_protein.pkl
